In [6]:
'''
Project — Movie Recommendation System (Content-Based Filtering)
Objective:
Build an AI system that recommends movies similar to a movie selected by the user based on movie features like genre, keywords, cast, and overview.
Example dataset:
| movie_id | title    | genre   | overview         |
| -------- | -------- | ------- | ---------------- |
| 1        | Batman   | Action  | Dark knight hero |
| 2        | Avengers | Action  | Superhero team   |
| 3        | Titanic  | Romance | Love story       |
| 4        | Superman | Action  | Alien hero       |

Concepts Used:

  Content-Based Filtering

  Text Feature Extraction

  Cosine Similarity

Input: Batman
Output: The Dark Knight
        Batman Begins
        Superman
        Avengers
'''

import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer # convert text into numbers
from sklearn.metrics.pairwise import cosine_similarity

# Sample dataset
data = {
    "title": ["Batman", "Avengers", "Titanic", "Superman"],
    "tags": [
        "action hero dark knight",
        "action superhero team",
        "romance love ship",
        "action alien hero"
    ]
}

movies = pd.DataFrame(data)

# Convert text to vectors
cv = CountVectorizer()
'''
 "action hero dark knight",
        "action superhero team",
        "romance love ship",
        "action alien hero"
  vocabulay = [action, hero, dark, knight, superhero, team, love, ship, alien]

'''
vectors = cv.fit_transform(movies['tags']).toarray()
'''
fit() -> Learn a vocabulary dictionary of all tokens in the raw documents.
transform() -> Transform documents to document-term matrix. -> converts text into vectors
ex: movie    action   hero  dark  knight  superhero  team
    Batman    1       1     1       1       0          0
    Avengers  1       1     0       0       1          1
    Titanic   0       0     1       0       1          0
    Superman  1       0     0       0       1          1

toarray() -> Convert to array
fit_transform () returns a sparse matrix. means mostly zero, .toarray() converts it into a normal NumPy array.

'''

# Calculate similarity
similarity = cosine_similarity(vectors)

'''
ex:      Batman   Avengers   Titanic   Superman

Batman    1.0     0.4         0.0       0.6

Avengers  0.4     1.0         0.0       0.6

Titanic   0.0     0.0         1.0       0.0

Superman  0.6     0.6         0.0       1.0
'''

# Recommendation function
def recommend(movie):
    index = movies[movies['title'] == movie].index[0]
    distances = similarity[index]
    '''
   Batman -> Batman -> 1
   Batman -> Avengers -> 0.4
   Batman -> Titanic -> 0.0
   Batman -> Superman -> 0.6
   [1.0,0.4,0.0,0.6]
    '''

    movie_list = sorted(list(enumerate(distances)),
                        reverse=True, # sort in descending order [(0,1.0),(3,0.6),(1,0.4),(2,0.0)]
                        key=lambda x: x[1])[1:4] # sort based on second value (similarity)

    '''
    enumerate(distances) -> adding index numbers to distances [(0,1.0),(1,0.4),(2,0.0),(3,0.6)]
    format -> (movie_index,similarity_score)
    '''


    for i in movie_list:
        print(movies.iloc[i[0]].title)

# Test
recommend("Batman")

Superman
Avengers
Titanic


In [10]:
'''Project — Product Recommendation System (E-Commerce)
Objective

Recommend similar products based on product description and category.

Example:

User views:

  Wireless Headphones

Recommended products:

  Bluetooth Earbuds
  Noise Cancelling Headphones
  Gaming Headset

Example Dtaset:
| product    | category    | description              |
| ---------- | ----------- | ------------------------ |
| Headphones | Electronics | wireless music audio     |
| Earbuds    | Electronics | bluetooth wireless sound |
| Laptop     | Electronics | computer portable device |
| Shoes      | Fashion     | running sports shoes     |


Concepts Used:

  Content-Based Filtering

  Text Processing

  Vectorization'''

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Dataset
data = {
    "product": ["Headphones", "Earbuds", "Laptop", "Shoes"],
    "description": [
        "wireless music audio",
        "bluetooth wireless sound",
        "portable computer device",
        "running sports footwear"
    ]
}

df = pd.DataFrame(data)

# Convert text to numbers
tfidf = TfidfVectorizer()
vectors = tfidf.fit_transform(df['description'])

# Similarity matrix
similarity = cosine_similarity(vectors)

# Recommendation function
def recommend(product_name):
    index = df[df['product'] == product_name].index[0]
    distances = similarity[index]

    product_list = sorted(list(enumerate(distances)),
                          reverse=True,
                          key=lambda x: x[1])[1:3]

    for i in product_list:
        print(df.iloc[i[0]].product)

recommend("Headphones")

<bound method Series.prod of product                         Earbuds
description    bluetooth wireless sound
Name: 1, dtype: object>
<bound method Series.prod of product                          Laptop
description    portable computer device
Name: 2, dtype: object>


In [9]:
'''
Project — User-Based Collaborative Filtering
Objective: Recommend movies to users based on similar users' preferences.

User-Item Matrix:
| User | Avengers | Titanic | Batman | Thor |
| ---- | -------- | ------- | ------ | ---- |
| A    | 5        | 0       | 4      | 0    |
| B    | 5        | 0       | 4      | 5    |
| C    | 0        | 5       | 0      | 0    |

Concepts Used:

  Collaborative Filtering

  User similarity

  Cosine similarity

Example:

      User A likes:

      Avengers
      Iron Man

      User B likes:

      Avengers
      Iron Man
      Thor

      System recommends:

      Thor
'''
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# User ratings
data = {
    "Avengers": [5,5,0],
    "Titanic": [0,0,5],
    "Batman": [4,4,0],
    "Thor": [0,5,0]
}

ratings = pd.DataFrame(data,
                       index=["UserA","UserB","UserC"])

print(ratings)

# Calculate user similarity
similarity = cosine_similarity(ratings)

'''
UserA vector = [5,0,4,0]
UserB vector = [5,0,4,5]
UserC vector = [0,5,0,0]

UserA -> UserA -> 1.0
UserA -> UserB -> 0.8
UserA -> UserC -> 0.0

'''

similarity_df = pd.DataFrame(similarity,
                             index=ratings.index,
                             columns=ratings.index)

print(similarity_df)

       Avengers  Titanic  Batman  Thor
UserA         5        0       4     0
UserB         5        0       4     5
UserC         0        5       0     0
         UserA    UserB  UserC
UserA  1.00000  0.78817    0.0
UserB  0.78817  1.00000    0.0
UserC  0.00000  0.00000    1.0
